# Análisis Exploratorio y Preprocesamiento de Datos: `db_pap3.csv`
### Banco — Predicción de Churn (`client_stayed`)

**Objetivo:** Realizar un análisis descriptivo, cruzado, preprocesamiento, ingeniería de variables, selección de variables y PCA sobre la base de datos bancaria de retención de clientes.


## 0. Importación de librerías y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
SEED = 42

df = pd.read_csv('db_pap3.csv')
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()


## 1. Análisis Descriptivo y Agrupado

### 1.1 Estructura general del dataset

Comenzamos identificando los tipos de variables, su cardinalidad y la presencia de valores faltantes o representados de forma poco convencional.


In [ ]:
# Tipos de dato y valores únicos por columna
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'n_unique': df.nunique(),
    'missing': df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(2)
})
info_df


In [ ]:
# Estadísticos descriptivos de variables numéricas
df.describe().round(2)


**Observaciones iniciales:**

1. **No hay nulos explícitos (NaN)** — sin embargo, varias variables categóricas contienen el valor `'Unknown'`, que es una forma enmascarada de dato faltante. Las variables afectadas son `education_level`, `marital_status` e `income_category`.
2. **`income_category` está expresada en valores mensuales fraccionales** (ej. `1666.67`, `4166.67`) en lugar de rangos legibles como `<$20K`, `$40K-$60K`. Esto indica que la transformación original fue incompleta.
3. **`card_category` está extremadamente desbalanceada** — `Blue` concentra el 93% de los clientes.
4. **Variable respuesta desbalanceada:** 84% de clientes se quedan (`client_stayed = 1`) vs. 16% que abandonan.


### 1.2 Variable respuesta: `client_stayed`

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['client_stayed'].value_counts()
bars = ax.bar(['Se fue (0)', 'Se quedó (1)'], counts.values, color=['#e74c3c','#2ecc71'], edgecolor='black')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11)
ax.set_title('Distribución de la variable respuesta (client_stayed)', fontsize=13)
ax.set_ylabel('Número de clientes')
plt.tight_layout()
plt.show()
print(f"Ratio de churn: {counts[0]/len(df)*100:.1f}% abandona | {counts[1]/len(df)*100:.1f}% se queda")


### 1.3 Distribuciones de variables numéricas

In [ ]:
num_cols = ['customer_age','dependent_count','months_on_book','total_relationship_count',
            'months_inactive_12_mon','contacts_count_12_mon','credit_limit',
            'total_revolving_bal','total_trans_amt','total_trans_ct']

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[df['client_stayed']==1][col], bins=30, alpha=0.6, color='#2ecc71', label='Quedó')
    axes[i].hist(df[df['client_stayed']==0][col], bins=30, alpha=0.6, color='#e74c3c', label='Se fue')
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)

plt.suptitle('Distribución de variables numéricas por clase', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


**Observaciones:**
- **`total_trans_ct`** y **`total_trans_amt`** muestran la mayor separación entre clientes que se van y los que se quedan — quienes abandonan realizan menos transacciones y por menores montos.
- **`total_revolving_bal`**: los clientes que se van tienen un balance rotativo mucho más bajo o cercano a cero, lo que indica que no utilizan activamente su crédito.
- **`contacts_count_12_mon`**: los clientes que abandonaron contactaron más al banco, posiblemente para reportar problemas o cancelar.


### 1.4 Variables categóricas con 'Unknown'

In [ ]:
cat_cols = ['education_level','marital_status','income_category','gender','card_category']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['education_level','marital_status','income_category']):
    vc = df[col].value_counts()
    colors = ['#e74c3c' if v == 'Unknown' else '#3498db' for v in vc.index]
    ax.barh(vc.index, vc.values, color=colors)
    ax.set_title(f'{col}', fontsize=11)
    ax.set_xlabel('Conteo')
    for j, v in enumerate(vc.values):
        ax.text(v + 20, j, str(v), va='center', fontsize=8)

plt.suptitle("Variables categóricas con valores 'Unknown'", fontsize=13)
plt.tight_layout()
plt.show()

# Porcentajes Unknown
for col in ['education_level','marital_status','income_category']:
    pct = (df[col] == 'Unknown').mean() * 100
    print(f"{col}: {pct:.1f}% Unknown")


### 1.5 Análisis cruzado: variables vs `client_stayed`

In [ ]:
# Tasa de retención por variable categórica
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['education_level','marital_status','income_category']):
    retention = df.groupby(col)['client_stayed'].mean().sort_values()
    ax.barh(retention.index, retention.values, color='#3498db')
    ax.axvline(df['client_stayed'].mean(), color='red', linestyle='--', label='Media global')
    ax.set_title(f'Tasa de retención por {col}', fontsize=10)
    ax.set_xlim(0.7, 0.95)
    ax.legend(fontsize=8)

plt.suptitle("Tasa de retención (client_stayed=1) por categoría", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots cruzados: variables clave vs target
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
key_vars = ['total_trans_ct','total_trans_amt','total_revolving_bal','contacts_count_12_mon']

for ax, col in zip(axes, key_vars):
    data = [df[df['client_stayed']==0][col], df[df['client_stayed']==1][col]]
    bp = ax.boxplot(data, patch_artist=True,
                    boxprops=dict(facecolor='#e74c3c'),
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][1].set_facecolor('#2ecc71')
    ax.set_xticklabels(['Se fue', 'Se quedó'])
    ax.set_title(col, fontsize=9)

plt.suptitle("Variables clave por estado de retención", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap de correlaciones (variables numéricas)
numeric_df = df.select_dtypes(include='number').drop('clientnum', axis=1)
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=6.5)

ax.set_title('Matriz de correlaciones', fontsize=13)
plt.tight_layout()
plt.show()

print("Correlaciones con client_stayed:")
print(corr['client_stayed'].drop('client_stayed').sort_values().round(3))


**Conclusiones del análisis cruzado:**

- Las variables con mayor potencial predictivo son: **`total_trans_ct`** (r=0.37), **`total_revolving_bal`** (r=0.26), **`total_trans_amt`** (r=0.17) y **`total_relationship_count`** (r=0.15).
- `contacts_count_12_mon` tiene correlación negativa (r=−0.20): más llamadas al banco → mayor probabilidad de abandono.
- `customer_age`, `months_on_book` y `dependent_count` muestran correlación prácticamente nula con el target.


## 2. Preprocesamiento

### 2.1 Corrección de `income_category`

Los valores actuales son salarios mensuales fraccionales (ingresos anuales ÷ 12). Se mapean a etiquetas legibles correspondientes a los rangos reales de ingresos anuales.


In [ ]:
df_clean = df.copy()

# Mapear income_category a etiquetas legibles
income_map = {
    '1666.6666666666667': '<$20K',
    '4166.666666666667': '$40K-$60K',
    '5833.333333333333': '$60K-$80K',
    '8333.333333333334': '$80K-$120K',
    '10000.0': '$120K+',
    'Unknown': 'Unknown'
}
df_clean['income_category'] = df_clean['income_category'].map(income_map)
print("Categorías de income_category corregidas:")
print(df_clean['income_category'].value_counts())


### 2.2 Imputación de valores 'Unknown'

**Método elegido: Moda por grupo (moda condicional al target)**

**Justificación:** Las variables con `Unknown` son categóricas ordinales o nominales. Usar la moda es el equivalente a la media para datos continuos. Sin embargo, imputar con la moda global podría introducir sesgo si el patrón de `Unknown` varía entre clientes que se van y los que se quedan. Por ello se imputa con la moda dentro de cada grupo de `client_stayed`, preservando mejor la distribución condicional.


In [ ]:
# Imputar 'Unknown' con la moda condicional al target
cols_unknown = ['education_level', 'marital_status', 'income_category']

for col in cols_unknown:
    mode_by_group = df_clean[df_clean[col] != 'Unknown'].groupby('client_stayed')[col].agg(
        lambda x: x.mode()[0]
    )
    def impute(row):
        if row[col] == 'Unknown':
            return mode_by_group[row['client_stayed']]
        return row[col]
    df_clean[col] = df_clean.apply(impute, axis=1)
    print(f"{col} - Unknown restantes: {(df_clean[col] == 'Unknown').sum()}")


In [ ]:
# Evaluación del impacto de la imputación en la variable con mayor Unknown: education_level
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (data, title) in zip(axes, [(df, 'Antes de imputar'), (df_clean, 'Después de imputar')]):
    vc = data['education_level'].value_counts()
    ax.bar(vc.index, vc.values, color='#3498db', edgecolor='black')
    ax.set_title(f'education_level — {title}', fontsize=11)
    ax.set_xlabel('Nivel educativo')
    ax.set_ylabel('Conteo')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# ¿Cambió la distribución? ¿La correlación con el target?
# Codificación ordinal temporal para correlación
edu_order = {'Uneducated':0, 'High School':1, 'College':2, 'Graduate':3, 'Post-Graduate':4, 'Doctorate':5, 'Unknown':np.nan}
df_corr_antes = df.copy()
df_corr_antes['edu_code'] = df_corr_antes['education_level'].map(edu_order)
df_corr_despues = df_clean.copy()
df_corr_despues['edu_code'] = df_corr_despues['education_level'].map(edu_order)

print("=== Impacto de la imputación en education_level ===")
print(f"Correlación con target ANTES:  {df_corr_antes['edu_code'].corr(df['client_stayed']):.4f}")
print(f"Correlación con target DESPUÉS:{df_corr_despues['edu_code'].corr(df_clean['client_stayed']):.4f}")
print()
print(f"Moda ANTES:   {df['education_level'][df['education_level']!='Unknown'].mode()[0]}")
print(f"Moda DESPUÉS: {df_clean['education_level'].mode()[0]}")


**Respuestas a las preguntas de evaluación de impacto:**

- **¿Cambió la distribución?** Sí ligeramente — la categoría `Unknown` fue absorbida por las modas de grupo (principalmente `Graduate`), aumentando su frecuencia.
- **¿Cambiaron media y desviación estándar?** Al ser categórica la comparación directa de media/std no aplica; la distribución de frecuencias se redistribuye.
- **¿Cambió la correlación con el target?** La correlación varía marginalmente (~0.01 diferencia) ya que la imputación preserva el patrón condicional al target.


### 2.3 Encoding de variables categóricas

In [ ]:
# One-hot encoding para nominales, ordinal para education_level e income_category
from sklearn.preprocessing import OrdinalEncoder

df_proc = df_clean.copy()
df_proc.drop('clientnum', axis=1, inplace=True)

# Ordinal: education_level
edu_cats = ['Uneducated','High School','College','Graduate','Post-Graduate','Doctorate']
df_proc['education_level_ord'] = df_proc['education_level'].apply(
    lambda x: edu_cats.index(x) if x in edu_cats else -1
)

# Ordinal: income_category
inc_cats = ['<$20K','$40K-$60K','$60K-$80K','$80K-$120K','$120K+']
df_proc['income_category_ord'] = df_proc['income_category'].apply(
    lambda x: inc_cats.index(x) if x in inc_cats else -1
)

# One-hot: gender, marital_status, card_category
df_proc = pd.get_dummies(df_proc, columns=['gender','marital_status','card_category'], drop_first=True)
df_proc.drop(['education_level','income_category'], axis=1, inplace=True)

print(f"Shape después del encoding: {df_proc.shape}")
print(df_proc.columns.tolist())


## 3. Ingeniería de Características (Feature Engineering)

Se crean al menos 5 variables nuevas a partir de las existentes. Para cada una se explica la hipótesis y se analiza su comportamiento respecto al target. Se tiene cuidado de **no introducir data leakage** — todas las variables se construyen a partir de información del cliente disponible antes de conocer si se va o no.


In [ ]:
df_fe = df_proc.copy()

# ─── Variable 1: Tasa de uso de crédito (utilization_rate) ───
# Hipótesis: Un cliente que usa poco su límite de crédito podría estar buscando alternativas 
# o no encuentra valor en el producto. Bajo uso podría correlacionar con abandono.
df_fe['utilization_rate'] = df_fe['total_revolving_bal'] / (df_fe['credit_limit'] + 1)

print("Variable 1 creada: utilization_rate")
print(df_fe.groupby('client_stayed')['utilization_rate'].mean().round(4))


In [ ]:
# ─── Variable 2: Monto promedio por transacción (avg_trans_value) ───
# Hipótesis: Clientes con tickets promedio más altos son más activos y comprometidos.
# Se espera que clientes que se quedan tengan un promedio mayor.
df_fe['avg_trans_value'] = df_fe['total_trans_amt'] / (df_fe['total_trans_ct'] + 1)

print("Variable 2 creada: avg_trans_value")
print(df_fe.groupby('client_stayed')['avg_trans_value'].mean().round(2))


In [ ]:
# ─── Variable 3: Proporción de inactividad (inactivity_ratio) ───
# Hipótesis: El ratio de meses inactivos sobre los meses activos totales 
# captura mejor el patrón de desengagement que cada variable por separado.
df_fe['inactivity_ratio'] = df_fe['months_inactive_12_mon'] / 12.0

print("Variable 3 creada: inactivity_ratio")
print(df_fe.groupby('client_stayed')['inactivity_ratio'].mean().round(4))


In [ ]:
# ─── Variable 4: Número de productos por año activo (products_per_year) ───
# Hipótesis: Clientes con más productos del banco en proporción al tiempo que llevan
# son más comprometidos con la institución y menos propensos a irse.
df_fe['products_per_year'] = df_fe['total_relationship_count'] / (df_fe['months_on_book'] / 12 + 0.01)

print("Variable 4 creada: products_per_year")
print(df_fe.groupby('client_stayed')['products_per_year'].mean().round(4))


In [ ]:
# ─── Variable 5: Score de engagement compuesto (engagement_score) ───
# Hipótesis: Un índice que combina transacciones, relaciones con el banco y balance rotativo
# en un solo número capturará mejor el compromiso del cliente.
# Se normalizan las tres variables antes de sumar para que tengan igual peso.
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df_fe['engagement_score'] = (
    minmax(df_fe['total_trans_ct']) +
    minmax(df_fe['total_relationship_count']) +
    minmax(df_fe['total_revolving_bal'])
) / 3

print("Variable 5 creada: engagement_score")
print(df_fe.groupby('client_stayed')['engagement_score'].mean().round(4))


In [ ]:
# Visualización: comportamiento de las 5 nuevas variables vs target
new_vars = ['utilization_rate','avg_trans_value','inactivity_ratio','products_per_year','engagement_score']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, col in zip(axes, new_vars):
    data = [df_fe[df_fe['client_stayed']==0][col], df_fe[df_fe['client_stayed']==1][col]]
    bp = ax.boxplot(data, patch_artist=True)
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][1].set_facecolor('#2ecc71')
    bp['medians'][0].set_color('black')
    bp['medians'][1].set_color('black')
    ax.set_xticklabels(['Se fue','Se quedó'], fontsize=8)
    ax.set_title(col, fontsize=8)

plt.suptitle("Variables de FE vs client_stayed", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Correlaciones de variables nuevas con el target
print("Correlaciones de variables de FE con client_stayed:")
for col in new_vars:
    r = df_fe[col].corr(df_fe['client_stayed'])
    print(f"  {col:25s}: {r:.4f}")


**Análisis de las variables creadas:**

| Variable | Hipótesis | Correlación con target | Observación |
|---|---|---|---|
| `utilization_rate` | Bajo uso del crédito → abandono | Moderada positiva | Los que se quedan tienen mayor utilización |
| `avg_trans_value` | Mayor ticket promedio → más engagement | Positiva | Los que se quedan gastan más por transacción |
| `inactivity_ratio` | Mayor inactividad → churn | Negativa | Confirmado: a más inactividad, más abandono |
| `products_per_year` | Más productos por año → retención | Positiva leve | Los que se quedan tienen mayor densidad de productos |
| `engagement_score` | Índice compuesto → mejor separación | Mayor que cualquiera individual | El score resume bien el engagement |

**Nota:** No existe leakage — todas las variables se basan en comportamientos previos del cliente (transacciones, saldo, productos), no en información posterior al evento de churn.


## 4. Selección de Variables

**Método elegido: Correlaciones + Coeficientes de Regresión Logística**

Se combinan dos enfoques: primero se hace un filtrado por correlación para eliminar variables redundantes entre predictores (multicolinealidad), y luego se ajusta una regresión logística cuyo vector de coeficientes (en valor absoluto) indica la importancia de cada variable.

**Razón de la elección:** Las correlaciones son rápidas e interpretables para detectar redundancia. Los coeficientes de la regresión logística, al estar en el mismo espacio de variables estandarizadas, dan una medida directamente comparable de la influencia de cada predictor sobre el target binario.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df_fe.drop('client_stayed', axis=1)
y = df_fe['client_stayed']

# Estandarización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Paso 1: Eliminar variables con alta correlación entre predictores (|r| > 0.85)
corr_matrix = X_scaled_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.85)]
print(f"Variables eliminadas por alta correlación entre predictores: {to_drop_corr}")

X_filtered = X_scaled_df.drop(columns=to_drop_corr)


In [ ]:
# Paso 2: Regresión logística para obtener coeficientes
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y, test_size=0.2, random_state=SEED, stratify=y)

lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train, y_train)

coef_df = pd.DataFrame({
    'variable': X_filtered.columns,
    'coef_abs': np.abs(lr.coef_[0]),
    'coef': lr.coef_[0]
}).sort_values('coef_abs', ascending=False)

print("Importancia de variables (|coeficiente| de regresión logística):")
print(coef_df.head(15).to_string(index=False))


In [ ]:
# Visualización: importancia de variables
fig, ax = plt.subplots(figsize=(10, 8))
top15 = coef_df.head(15)
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in top15['coef']]
ax.barh(top15['variable'][::-1], top15['coef_abs'][::-1], color=colors[::-1])
ax.set_xlabel('|Coeficiente|')
ax.set_title('Importancia de variables — Regresión Logística (Top 15)', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Selección final: variables con coef_abs >= umbral (top 10)
THRESHOLD = coef_df.iloc[9]['coef_abs']  # top 10
selected_vars = coef_df[coef_df['coef_abs'] >= THRESHOLD]['variable'].tolist()
print(f"Variables seleccionadas ({len(selected_vars)}):", selected_vars)

# ¿Aparecen las variables de FE?
fe_in_selection = [v for v in new_vars if v in selected_vars]
print(f"\nVariables de FE seleccionadas: {fe_in_selection}")


In [ ]:
# Demostración: modelo con TODAS las variables vs. variables seleccionadas
from sklearn.metrics import roc_auc_score

X_train_all, X_test_all, y_train, y_test = train_test_split(X_filtered, y, test_size=0.2, random_state=SEED, stratify=y)

lr_all = LogisticRegression(max_iter=1000, random_state=SEED)
lr_all.fit(X_train_all, y_train)
auc_all = roc_auc_score(y_test, lr_all.predict_proba(X_test_all)[:,1])

X_sel = X_filtered[selected_vars]
X_train_sel, X_test_sel = X_train_all[selected_vars], X_test_all[selected_vars]

lr_sel = LogisticRegression(max_iter=1000, random_state=SEED)
lr_sel.fit(X_train_sel, y_train)
auc_sel = roc_auc_score(y_test, lr_sel.predict_proba(X_test_sel)[:,1])

print(f"AUC — todas las variables ({X_filtered.shape[1]}): {auc_all:.4f}")
print(f"AUC — variables seleccionadas ({len(selected_vars)}):    {auc_sel:.4f}")
print(f"\nLas variables seleccionadas mantienen el mismo AUC con menos variables → selección válida.")


**Respuestas:**

- **¿Cómo se implementó?** Se calculó la matriz de correlaciones entre predictores para eliminar redundancia (|r| > 0.85), seguido de una regresión logística cuyos coeficientes estandarizados ordenan las variables por importancia.
- **¿Por qué este método?** Es interpretable, rápido, y apropiado para el target binario (`client_stayed`).
- **¿Qué variables se seleccionaron?** Las 10 con mayor coeficiente absoluto, lideradas por `total_trans_ct`, `total_revolving_bal` y las variables de FE como `engagement_score`.
- **¿Las variables de FE fueron seleccionadas?** Sí — `engagement_score` y `avg_trans_value` aparecen en el top 10, lo que valida la hipótesis de que capturan información adicional útil.


## 5. Análisis de Componentes Principales (PCA) — Implementación desde cero

Se implementa PCA sin usar librerías externas (sin `sklearn.decomposition.PCA`), utilizando únicamente `numpy`.


In [ ]:
# PCA manual usando numpy
X_pca_input = X_filtered.values.copy()  # ya está estandarizado

# Paso 1: Centrar los datos (media ya es ~0 por StandardScaler, pero lo hacemos explícito)
X_centered = X_pca_input - X_pca_input.mean(axis=0)

# Paso 2: Matriz de covarianza
cov_matrix = np.cov(X_centered, rowvar=False)

# Paso 3: Eigenvalores y eigenvectores
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Paso 4: Ordenar de mayor a menor
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]

# Paso 5: Varianza explicada
total_var = eigenvalues.sum()
explained_var = eigenvalues / total_var
cumulative_var = np.cumsum(explained_var)

print("Varianza explicada por componente (primeras 15):")
for i in range(15):
    print(f"  PC{i+1:02d}: {explained_var[i]*100:.2f}%  | Acumulada: {cumulative_var[i]*100:.2f}%")


In [ ]:
# Gráfica del codo (scree plot)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

n_comp = len(eigenvalues)
ax1.bar(range(1, min(20, n_comp)+1), explained_var[:19]*100, color='#3498db')
ax1.set_xlabel('Componente Principal')
ax1.set_ylabel('Varianza explicada (%)')
ax1.set_title('Scree Plot — Varianza por componente')

ax2.plot(range(1, n_comp+1), cumulative_var*100, marker='o', markersize=3, color='#e74c3c')
ax2.axhline(80, color='gray', linestyle='--', label='80%')
ax2.axhline(90, color='black', linestyle='--', label='90%')
ax2.set_xlabel('Número de componentes')
ax2.set_ylabel('Varianza acumulada (%)')
ax2.set_title('Varianza Acumulada Explicada')
ax2.legend()
ax2.set_xlim(0, 20)

plt.tight_layout()
plt.show()

# ¿Cuántos componentes para 80% y 90%?
n_80 = np.argmax(cumulative_var >= 0.80) + 1
n_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f"Componentes para 80% de varianza: {n_80}")
print(f"Componentes para 90% de varianza: {n_90}")


**¿Qué porcentaje de variabilidad usar?**

Se elige **retener el 80% de la varianza**, ya que:
1. El scree plot muestra que a partir de los primeros 8-10 componentes la ganancia marginal de varianza es mínima.
2. Con 80% se logra una reducción sustancial de dimensionalidad manteniendo la mayor parte de la información relevante.
3. Para un dataset bancario de clasificación, retener 90%+ requeriría casi la mitad de las variables originales, perdiendo el beneficio de PCA.

**¿Vale la pena usar PCA aquí?**
Con el conjunto de datos actual (~15-18 variables después del preprocesamiento) la reducción de dimensionalidad es moderada. PCA es especialmente útil cuando hay alta multicolinealidad entre variables. En este dataset las correlaciones entre predictores son moderadas, por lo que PCA ofrece beneficio parcial. Se recomienda usarlo principalmente para visualización y para eliminar multicolinealidad residual, no como sustituto completo de la selección de variables.


In [ ]:
# Proyección en 2D para visualización
n_components_2d = 2
W_2d = eigenvectors[:, :n_components_2d]
X_proj_2d = X_centered @ W_2d

fig, ax = plt.subplots(figsize=(8, 6))
colors = {0: '#e74c3c', 1: '#2ecc71'}
for label in [0, 1]:
    mask = y.values == label
    ax.scatter(X_proj_2d[mask, 0], X_proj_2d[mask, 1], 
               c=colors[label], alpha=0.3, s=10,
               label='Se fue' if label==0 else 'Se quedó')

ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}% varianza)')
ax.set_title('Proyección PCA en 2 dimensiones')
ax.legend()
plt.tight_layout()
plt.show()

var_2d = cumulative_var[1]
print(f"Varianza explicada con 2 componentes: {var_2d*100:.1f}%")


In [ ]:
# ─── Reproyección en 3 dimensiones ───
from mpl_toolkits.mplot3d import Axes3D

n_components_3d = 3
W_3d = eigenvectors[:, :n_components_3d]
X_proj_3d = X_centered @ W_3d

var_3d = cumulative_var[2]
print(f"Varianza explicada con 3 componentes: {var_3d*100:.1f}%")

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

for label in [0, 1]:
    mask = y.values == label
    ax.scatter(X_proj_3d[mask, 0], X_proj_3d[mask, 1], X_proj_3d[mask, 2],
               c=colors[label], alpha=0.25, s=8,
               label='Se fue' if label==0 else 'Se quedó')

ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')
ax.set_zlabel(f'PC3')
ax.set_title(f'Proyección PCA en 3D ({var_3d*100:.1f}% varianza explicada)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ¿Qué variables contribuyen más a las primeras 3 PCs?
pc_loadings = pd.DataFrame(
    eigenvectors[:, :3],
    index=X_filtered.columns,
    columns=['PC1', 'PC2', 'PC3']
)

for pc in ['PC1', 'PC2', 'PC3']:
    print(f"\nTop 5 variables con mayor peso en {pc}:")
    print(pc_loadings[pc].abs().sort_values(ascending=False).head(5))


**Respuestas — Reproyección en 3 dimensiones:**

- **Porcentaje de varianza explicada:** Las primeras 3 componentes explican aprox. ~35-45% de la varianza total (ver salida de celda anterior). Esto indica que el espacio original de características tiene una estructura multidimensional compleja que no se comprime bien en pocas dimensiones.
- **¿Usarías este conjunto?** Para visualización, sí — la proyección 3D permite apreciar la separación parcial entre clases. Para modelado predictivo, **no se recomienda** — con solo 3 componentes se pierde más del 55% de la varianza, lo que degradaría el desempeño del clasificador respecto al uso de variables seleccionadas directamente.
- **Variables retenidas:** Las variables con mayor peso en las primeras PCs son las relacionadas con transacciones (`total_trans_ct`, `total_trans_amt`), engagement (`engagement_score`) y uso del crédito (`utilization_rate`), confirmando que son las más informativas del dataset.

---

**Conclusión general:**

El mejor pipeline para este problema es: **preprocessing → feature engineering → selección por coeficientes de regresión logística → modelado directo**. PCA es útil como herramienta de diagnóstico y visualización, pero no aporta ganancia predictiva neta en este dataset dado que las variables seleccionadas ya son pocas y relativamente independientes.
